# 🎵 Free AI Song Studio on Kaggle: ACE-Step 1.5 **XL**

Full songs with vocals from your lyrics and a style description, using the **XL (4B)** model.
XL sounds better than the 2B model in the Colab notebook. It needs ~20 GB of RAM to load, which Kaggle's free tier has (~29 GB) and free Colab doesn't (~12.7 GB).

### One-time setup
1. On Kaggle: **Create → New Notebook**, then **File → Import Notebook** and upload this `.ipynb`.
2. In the right panel, open **Settings** (⋮ → *Notebook options* on newer layouts):
   - **Accelerator → GPU T4 x2**. Don't pick P100: ACE-Step's PyTorch build doesn't support it.
   - **Internet → On**. This needs a phone-verified Kaggle account.
3. Run the cells top to bottom with **Shift + Enter**.

| Step | What it does | Time |
|---|---|---|
| 1 | Settings + checks (GPU, internet, disk) | seconds |
| 2 | Install ACE-Step and download the models (~30 GB) | ~10–20 min per session |
| 3 | Start the song engine (loads XL once) | ~3–8 min |
| 4 | **Make a song**: several takes, played right here | a few min per run |
| 5 | Fix one section of a take (Repaint) | a few min |
| 6 | Master your favorite take | ~1 min per song |
| 7 | Download everything as a zip | seconds |

Everything runs inside this notebook, with no public links or tunnels, which keeps it within Kaggle's rules for personal use.
Free GPU time is about 30 h per week, so **stop the session** (power icon, top right) when you're done.

In [ ]:
# 1️⃣ Settings + checks. Edit these values if you like, then run (Shift+Enter).
SONG_MODEL = "acestep-v15-xl-turbo"  # "acestep-v15-xl-turbo" (best, recommended) | "acestep-v15-xl-sft" (slower, 50 steps) | "acestep-v15-turbo" (2B, smaller)
LM_MODEL = "acestep-5Hz-lm-1.7B"     # song planner; use "acestep-5Hz-lm-0.6B" only if you get out-of-memory errors
LM_BACKEND = "pt"                    # "pt" is reliable on T4; "vllm" is faster but can hang when GPU memory runs low
ACESTEP_VERSION = "pinned"           # "pinned" (what this notebook was written against) or "latest"

import os
import shutil
import subprocess
import urllib.request

PINNED_COMMIT = "ca1e85fe9430179831e6bc6be790c332190a3866"  # ACE-Step-1.5 main, 2026-08-29
OUT_DIR = "/kaggle/working/songs"  # shows up in the Output panel; downloadable
NEEDED_GB = {"acestep-v15-xl-turbo": 45, "acestep-v15-xl-sft": 45, "acestep-v15-turbo": 25}

# GPU
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True)
if gpu.returncode != 0:
    raise RuntimeError("No GPU attached. Settings → Accelerator → GPU T4 x2, then run this cell again.")
gpu_names = gpu.stdout.strip().splitlines()
print("GPU:", " | ".join(gpu_names))
if any("P100" in name for name in gpu_names):
    raise RuntimeError("P100 isn't supported by ACE-Step's PyTorch build. Settings → Accelerator → GPU T4 x2.")

# Internet
try:
    urllib.request.urlopen("https://huggingface.co", timeout=15)
except Exception:
    raise RuntimeError("No internet. Settings → Internet → On (needs a phone-verified Kaggle account).")

# Disk: /kaggle/working is capped at 20 GB, so install in the roomiest scratch location instead.
bases = []
for base in ["/kaggle/temp", "/tmp", "/root"]:
    try:
        os.makedirs(base, exist_ok=True)
        bases.append((shutil.disk_usage(base).free / 1e9, base))
    except OSError:
        pass
existing = [b for _, b in bases if os.path.isdir(f"{b}/acestep/ACE-Step-1.5/.venv")]
if existing:
    base = existing[0]
    print(f"Found an existing install in {base}/acestep. Reusing it.")
else:
    free_gb, base = max(bases)
    if free_gb < NEEDED_GB[SONG_MODEL] and SONG_MODEL != "acestep-v15-turbo":
        print(f"⚠️ Only {free_gb:.0f} GB of scratch disk; XL needs ~{NEEDED_GB[SONG_MODEL]} GB. "
              "Using the 2B turbo model instead.")
        SONG_MODEL = "acestep-v15-turbo"
    if free_gb < NEEDED_GB[SONG_MODEL]:
        raise RuntimeError(f"Only {free_gb:.0f} GB of scratch disk free; need ~{NEEDED_GB[SONG_MODEL]} GB.")
    print(f"Scratch disk: {free_gb:.0f} GB free in {base}")

WORK_DIR = f"{base}/acestep"
REPO_DIR = f"{WORK_DIR}/ACE-Step-1.5"
JOBS_DIR = f"{WORK_DIR}/jobs"
for d in (WORK_DIR, JOBS_DIR, OUT_DIR):
    os.makedirs(d, exist_ok=True)

if not shutil.which("ffmpeg"):
    subprocess.run("apt-get -qq update && apt-get -qq install -y ffmpeg", shell=True, check=True)

print(f"Song model: {SONG_MODEL} | planner: {LM_MODEL} | songs saved to {OUT_DIR}")

In [ ]:
# 2️⃣ Install ACE-Step 1.5 + download the models (~10–20 min)
import os
import subprocess


def sh(cmd, cwd=None):
    print(f"$ {cmd}", flush=True)
    env = dict(os.environ, MPLBACKEND="Agg", UV_NO_PROGRESS="1", PYTHONUNBUFFERED="1", UV_CACHE_DIR=f"{WORK_DIR}/uv-cache")
    p = subprocess.Popen(cmd, shell=True, cwd=cwd, env=env, text=True,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    for line in p.stdout:
        print(line, end="")
    if p.wait() != 0:
        raise RuntimeError(f"Command failed: {cmd}")


if not os.path.isdir(f"{REPO_DIR}/.git"):
    sh(f"git clone https://github.com/ace-step/ACE-Step-1.5.git {REPO_DIR}")
if ACESTEP_VERSION == "pinned":
    sh(f"git fetch --quiet origin {PINNED_COMMIT} && git checkout --quiet {PINNED_COMMIT}", cwd=REPO_DIR)
else:
    sh("git fetch --quiet origin main && git checkout --quiet origin/main", cwd=REPO_DIR)

# A T4 has no bfloat16, so ACE-Step runs the song model in float16 there, which turns every
# sample into NaN. Run it in float32 instead (int8 quantization keeps the weights small).
_orch = f"{REPO_DIR}/acestep/core/generation/handler/init_service_orchestrator.py"
_src = open(_orch).read()
if "self.dtype = torch.float16" in _src:
    open(_orch, "w").write(_src.replace("self.dtype = torch.float16", "self.dtype = torch.float32", 1))
    print("Patched ACE-Step: the song model runs in float32 on this GPU.")
elif "self.dtype = torch.float32" not in _src:
    raise RuntimeError("Couldn't apply the float32 patch; ACE-Step's code changed. Use ACESTEP_VERSION = \"pinned\".")

# uv builds an isolated Python environment with the exact versions ACE-Step was tested with.
sh("pip install -q uv")
sh("uv sync", cwd=REPO_DIR)

# Main bundle: planner (1.7B), audio decoder (VAE), text encoder (+ the 2B turbo model).
sh("uv run acestep-download", cwd=REPO_DIR)
if SONG_MODEL != "acestep-v15-turbo":
    sh(f"uv run acestep-download --model {SONG_MODEL}", cwd=REPO_DIR)  # XL: ~20 GB
if LM_MODEL != "acestep-5Hz-lm-1.7B":
    sh(f"uv run acestep-download --model {LM_MODEL}", cwd=REPO_DIR)

print("\n✅ Installed. Run step 3.")

In [ ]:
# 3️⃣ Start the song engine. It loads the models once and keeps them ready for steps 4 and 5.
import glob
import json
import os
import subprocess
import time
import uuid

from IPython.display import Audio, display

WORKER_SRC = r'''
# Runs inside ACE-Step's own Python environment. Loads the models once, then turns
# job files from the notebook into songs. Mirrors the official launcher's setup.
import glob, json, os, sys, time, traceback

REPO_DIR, JOBS_DIR, DIT_MODEL, LM_MODEL, LM_BACKEND = sys.argv[1:6]
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

from acestep.gpu_config import (VRAM_AUTO_OFFLOAD_THRESHOLD_GB, get_gpu_config,
                                resolve_lm_backend, set_global_gpu_config)
from acestep.handler import AceStepHandler
from acestep.inference import GenerationConfig, GenerationParams, generate_music
from acestep.llm_inference import LLMHandler

gpu_config = get_gpu_config()
set_global_gpu_config(gpu_config)
vram_gb = gpu_config.gpu_memory_gb
offload = 0 < vram_gb < VRAM_AUTO_OFFLOAD_THRESHOLD_GB
quantization = "int8_weight_only" if gpu_config.quantization_default else None
backend = resolve_lm_backend(LM_BACKEND, gpu_config)
print(f"[engine] GPU {vram_gb:.1f} GB | offload={offload} | "
      f"offload_dit={gpu_config.offload_dit_to_cpu_default} | quantization={quantization} | "
      f"planner backend={backend}", flush=True)

dit = AceStepHandler()
status, ok = dit.initialize_service(
    project_root=REPO_DIR, config_path=DIT_MODEL, device="auto",
    use_flash_attention=dit.is_flash_attention_available("auto"), compile_model=False,
    offload_to_cpu=offload, offload_dit_to_cpu=gpu_config.offload_dit_to_cpu_default,
    quantization=quantization)
if not ok:
    print(f"[engine] FAILED to load the song model: {status}", flush=True)
    sys.exit(1)

print(f"[engine] song model dtype={dit.dtype}", flush=True)

llm = LLMHandler()
status, ok = llm.initialize(checkpoint_dir=os.path.join(REPO_DIR, "checkpoints"), lm_model_path=LM_MODEL,
                            backend=backend, device="auto", offload_to_cpu=offload, dtype=None)
planner = llm if ok and llm.llm_initialized else None
if planner is None:
    print(f"[engine] WARNING: planner failed to load, songs will be made without it: {status}", flush=True)

IS_TURBO = "turbo" in DIT_MODEL
print("[engine] READY", flush=True)


def first(*values):
    for v in values:
        if v not in (None, "", 0, -1, -1.0):
            return v
    return None


def make(job):
    p = dict(job["params"])
    p.setdefault("shift", 3.0)  # the studio UI's default; the library default (1.0) is weaker for turbo
    p.setdefault("inference_steps", 8 if IS_TURBO else 50)
    params = GenerationParams(**p)
    config = GenerationConfig(batch_size=int(job["takes"]), use_random_seed=True, audio_format="flac")
    os.makedirs(job["save_dir"], exist_ok=True)
    result = generate_music(dit, planner, params=params, config=config, save_dir=job["save_dir"])
    audios = []
    for a in result.audios:
        ap = a.get("params") or {}
        audios.append({"path": a.get("path"), "seed": ap.get("seed"),
                       "bpm": first(ap.get("cot_bpm"), ap.get("bpm")),
                       "keyscale": first(ap.get("cot_keyscale"), ap.get("keyscale")),
                       "duration": first(ap.get("cot_duration"), ap.get("duration"))})
    ok = result.success and any(x["path"] for x in audios)
    return {"success": ok, "error": None if ok else (result.error or result.status_message), "audios": audios}


while True:
    pending = sorted(glob.glob(os.path.join(JOBS_DIR, "*.job.json")))
    if not pending:
        time.sleep(1)
        continue
    job_path = pending[0]
    with open(job_path) as f:
        job = json.load(f)
    os.remove(job_path)
    started = time.time()
    print(f"[engine] job {job['id']} started", flush=True)
    try:
        out = make(job)
    except Exception as exc:
        traceback.print_exc()
        message = f"{type(exc).__name__}: {exc}"
        if "out of memory" in message.lower():
            message += " | Try fewer takes or a shorter duration."
        out = {"success": False, "error": message, "audios": []}
        try:
            import torch
            torch.cuda.empty_cache()
        except Exception:
            pass
    out["seconds"] = round(time.time() - started, 1)
    tmp = os.path.join(JOBS_DIR, job["id"] + ".result.tmp")
    with open(tmp, "w") as f:
        json.dump(out, f, default=str)
    os.replace(tmp, os.path.join(JOBS_DIR, job["id"] + ".result.json"))
    print(f"[engine] job {job['id']} finished in {out['seconds']}s", flush=True)
'''

ENGINE_LOG = f"{WORK_DIR}/engine.log"
WORKER_PY = f"{WORK_DIR}/ace_worker.py"
with open(WORKER_PY, "w") as f:
    f.write(WORKER_SRC)

# Stop an engine left over from an earlier run so it doesn't hold GPU memory.
if "engine_proc" in globals() and engine_proc.poll() is None:
    engine_proc.terminate()
    engine_proc.wait(timeout=60)
subprocess.run(["pkill", "-f", WORKER_PY], check=False)
for stale in glob.glob(os.path.join(JOBS_DIR, "*")):
    os.remove(stale)

engine_proc = subprocess.Popen(
    [f"{REPO_DIR}/.venv/bin/python", WORKER_PY, REPO_DIR, JOBS_DIR, SONG_MODEL, LM_MODEL, LM_BACKEND],
    cwd=REPO_DIR, stdout=open(ENGINE_LOG, "w"), stderr=subprocess.STDOUT,
    env=dict(os.environ, MPLBACKEND="Agg", PYTHONUNBUFFERED="1", CUDA_VISIBLE_DEVICES="0", ACESTEP_PROJECT_ROOT=REPO_DIR),
)


def engine_log(n=1):
    with open(ENGINE_LOG, errors="ignore") as f:
        lines = [l.rstrip() for l in f if l.strip()]
    return "\n".join(lines[-n:])


def wait_for(done, what, timeout_min):
    started, last_note = time.time(), time.time()
    while not done():
        time.sleep(3)
        if engine_proc.poll() is not None:
            print(engine_log(60))
            raise RuntimeError("The song engine stopped (log above). Fix the setting it mentions in step 1, "
                               "then run steps 1 and 3 again.")
        if time.time() - last_note > 30:
            print(f"  … {int(time.time() - started)}s  {engine_log()[:150]}", flush=True)
            last_note = time.time()
        if time.time() - started > timeout_min * 60:
            raise TimeoutError(f"{what} took longer than {timeout_min} min. Engine log:\n{engine_log(30)}")


def run_ffmpeg(*args):
    r = subprocess.run(["ffmpeg", "-y", "-loglevel", "error", *args], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"ffmpeg failed:\n{r.stderr[-2000:]}")


def to_mp3(src, dst, bitrate="320k"):
    run_ffmpeg("-i", src, "-c:a", "libmp3lame", "-b:a", bitrate, dst)


def run_job(params, takes, label):
    job_id = time.strftime("%Y%m%d-%H%M%S") + "-" + uuid.uuid4().hex[:4]
    job = {"id": job_id, "takes": max(1, min(int(takes), 4)), "params": params,
           "save_dir": os.path.join(OUT_DIR, "takes", job_id)}
    tmp = os.path.join(JOBS_DIR, job_id + ".tmp")
    with open(tmp, "w") as f:
        json.dump(job, f)
    os.replace(tmp, os.path.join(JOBS_DIR, job_id + ".job.json"))
    result_path = os.path.join(JOBS_DIR, job_id + ".result.json")
    print(f"{label}…", flush=True)
    wait_for(lambda: os.path.exists(result_path), label, timeout_min=75)
    with open(result_path) as f:
        result = json.load(f)
    if not result["success"]:
        raise RuntimeError(f"Generation failed: {result['error']}")
    print(f"✅ Done in {result['seconds']:.0f}s\n")
    paths = []
    for i, a in enumerate(result["audios"], 1):
        if not a["path"]:
            continue
        preview = os.path.splitext(a["path"])[0] + "_preview.mp3"
        to_mp3(a["path"], preview, "192k")
        details = [f"{a['bpm']} BPM" if a.get("bpm") else "", a.get("keyscale") or "",
                   f"{float(a['duration']):.0f}s" if a.get("duration") else "", f"seed {a.get('seed')}"]
        print(f"Take {i}:  " + " · ".join(d for d in details if d))
        display(Audio(filename=preview))
        paths.append(a["path"])
    return paths


print("Loading the models (first start: ~3–8 min)…", flush=True)
wait_for(lambda: "[engine] READY" in engine_log(400), "Loading the models", timeout_min=30)
print("\n" + "\n".join(l for l in engine_log(400).splitlines() if l.startswith("[engine]")))
print("\n✅ Engine ready. Go to step 4.")

## ✍️ How to get Suno-like results

**Caption (the style):** describe the track the way a producer would. Include genre and sub-genre, tempo feel, instruments, vocal type, mood and mix ("polished, wide, radio-ready"). Don't use artist names, and avoid contradictions ("fast slow").

**Lyrics:** mark the structure with `[Intro]`, `[Verse 1]`, `[Pre-Chorus]`, `[Chorus]`, `[Bridge]`, `[Outro]`. Instrument cues work too, for example `[Instrumental Break - Synth Solo]`. Keep lines short and singable, and use one language per song. For an instrumental, set the lyrics to `[Instrumental]`.

**Habits that improve quality the most:**
1. **Make several takes and keep the best.** Picking from more takes is the single biggest lever: it's how YuE2 beat Suno v5 in its own benchmark.
2. **Fix a weak section with step 5** instead of starting over.
3. **Master your favorite take with step 6**.

In [ ]:
# 4️⃣ Make a song. Edit the text, then run. Each take plays below when it's done.
CAPTION = '''Modern synth-pop at 118 BPM with a four-on-the-floor beat, warm analog bass,
shimmering arpeggiated synths and gated reverb snares. Breathy, intimate female lead vocal
with stacked harmonies in the chorus. Bittersweet and nostalgic, polished, wide, radio-ready mix.'''

LYRICS = '''[Intro]

[Verse 1]
City lights are fading on the train ride home
Your voice still in my headphones on a dying phone
Every station flickers like a memory
Every empty seat is where you used to be

[Pre-Chorus]
And I know, and I know
It's time to let you go

[Chorus]
We were neon in the dark, we were louder than the rain
We were dancing on the edge of a never-ending train
Now the colors fade to grey but I still hear the beat
Of a city that was ours, of a heart that couldn't sleep

[Verse 2]
Coffee cups and postcards on the window sill
Every song we sang is playing still
I keep walking past the places that we knew
Every corner of this town still sounds like you

[Pre-Chorus]
And I know, and I know
It's time to let you go

[Chorus]
We were neon in the dark, we were louder than the rain
We were dancing on the edge of a never-ending train
Now the colors fade to grey but I still hear the beat
Of a city that was ours, of a heart that couldn't sleep

[Bridge]
Turn the lights down low
Let the echoes go

[Outro]'''

LANGUAGE = "en"   # en, es, fr, de, it, pt, ja, ko, zh, hi, ... ("unknown" = auto-detect)
TAKES = 2         # 1–4; the engine lowers this automatically if GPU memory runs short
DURATION = -1     # seconds (10–480), or -1 to let the planner choose from the lyrics
BPM = None        # e.g. 118, or None to let the planner choose
KEY = ""          # e.g. "A minor", or "" to let the planner choose

LAST_REQUEST = {"caption": CAPTION.strip(), "lyrics": LYRICS.strip(), "vocal_language": LANGUAGE,
                "duration": float(DURATION), "bpm": BPM, "keyscale": KEY, "thinking": True}
LAST_TAKES = run_job(LAST_REQUEST, TAKES, "Planning, composing and singing your song")

In [ ]:
# 5️⃣ (Optional) Fix one section of a take: regenerate START→END seconds and keep the rest.
TAKE_TO_FIX = 1        # take number from the last run above
START_SECONDS = 30.0
END_SECONDS = 45.0
NEW_CAPTION = ""       # optional: describe how this part should sound; empty = reuse the song's caption
TAKES = 2

if "LAST_TAKES" not in globals():
    raise RuntimeError("Make a song first (step 4).")
fix_request = {
    "task_type": "repaint",
    "src_audio": LAST_TAKES[TAKE_TO_FIX - 1],
    "repainting_start": float(START_SECONDS),
    "repainting_end": float(END_SECONDS),
    "chunk_mask_mode": "explicit",
    "caption": NEW_CAPTION.strip() or LAST_REQUEST["caption"],
    "lyrics": LAST_REQUEST["lyrics"],
    "vocal_language": LAST_REQUEST["vocal_language"],
}
LAST_TAKES = run_job(fix_request, TAKES, f"Repainting {START_SECONDS:.0f}s–{END_SECONDS:.0f}s of take {TAKE_TO_FIX}")
print("These fixed takes are now the 'last run' for steps 5 and 6.")

In [ ]:
# 6️⃣ Master your songs: loud and polished like a release.
#   "reference": match the loudness + tone of a professional song you add as input (best).
#                Right panel → Add Input → Upload → drop the song → Create. It appears under /kaggle/input.
#   "loudness":  streaming-standard -14 LUFS, no upload needed.
#   "none":      copy as-is.
MASTERING = "reference"
WHICH_TAKES = "all"    # "all" takes from the last run, or a list of take numbers like [2]
REFERENCE_FILE = ""    # optional exact path; otherwise the first audio file under /kaggle/input is used

import glob
import json
import os
import shutil
import subprocess

from IPython.display import Audio, display

AUDIO_EXT = (".mp3", ".wav", ".flac", ".m4a", ".ogg", ".aac")


def to_wav(src, dst, rate=44100):
    run_ffmpeg("-i", src, "-ar", str(rate), "-ac", "2", "-c:a", "pcm_s24le", dst)


def loudnorm(src, dst, target_lufs=-14.0, true_peak=-1.0, lra=11.0):
    # Two-pass EBU R128 loudness normalization: measure first, then apply a linear gain.
    base = f"loudnorm=I={target_lufs}:TP={true_peak}:LRA={lra}"
    probe = subprocess.run(["ffmpeg", "-hide_banner", "-i", src, "-af", f"{base}:print_format=json",
                            "-f", "null", "-"], capture_output=True, text=True)
    m = json.loads(probe.stderr[probe.stderr.rindex("{"):])
    af = (f"{base}:measured_I={m['input_i']}:measured_TP={m['input_tp']}:measured_LRA={m['input_lra']}"
          f":measured_thresh={m['input_thresh']}:offset={m['target_offset']}:linear=true")
    run_ffmpeg("-i", src, "-af", af, "-ar", "48000", "-c:a", "pcm_s24le", dst)


if "LAST_TAKES" not in globals() or not LAST_TAKES:
    raise RuntimeError("Make a song first (step 4).")
chosen = LAST_TAKES if WHICH_TAKES == "all" else [LAST_TAKES[i - 1] for i in WHICH_TAKES]

master_dir = os.path.join(OUT_DIR, "mastered")
work_dir = os.path.join(WORK_DIR, "mastering_tmp")
os.makedirs(master_dir, exist_ok=True)
os.makedirs(work_dir, exist_ok=True)

reference_wav = None
if MASTERING == "reference":
    reference = REFERENCE_FILE or next(iter(sorted(
        p for p in glob.glob("/kaggle/input/**/*", recursive=True) if p.lower().endswith(AUDIO_EXT))), None)
    if not reference:
        raise RuntimeError("No reference song found under /kaggle/input. Right panel → Add Input → Upload → "
                           "drop a professionally mastered song → Create, then run this cell again. "
                           "Or set MASTERING = \"loudness\".")
    print("Reference:", reference)
    subprocess.run(["pip", "install", "-q", "matchering"], check=True)
    import matchering as mg
    reference_wav = os.path.join(work_dir, "reference.wav")
    to_wav(reference, reference_wav)

for song in chosen:
    stem = f"{os.path.basename(os.path.dirname(song))}_{os.path.splitext(os.path.basename(song))[0]}"
    if MASTERING == "reference":
        final = os.path.join(master_dir, f"{stem}_master.wav")
        target_wav = os.path.join(work_dir, f"{stem}_in.wav")
        to_wav(song, target_wav)
        mg.process(target=target_wav, reference=reference_wav, results=[mg.pcm24(final)])
    elif MASTERING == "loudness":
        final = os.path.join(master_dir, f"{stem}_master.wav")
        loudnorm(song, final)
    else:
        final = os.path.join(master_dir, stem + os.path.splitext(song)[1])
        shutil.copy2(song, final)
    mp3 = os.path.splitext(final)[0] + ".mp3"
    to_mp3(final, mp3)
    print(f"✅ {os.path.basename(mp3)}  +  {os.path.basename(final)}")
    display(Audio(filename=mp3))

In [ ]:
# 7️⃣ Download your songs as one zip.
#    Files in /kaggle/working are lost when the session ends unless you download them first.
ZIP_WHAT = "mastered"   # "mastered" (just the finished songs) or "everything" (all takes too)

import os
import shutil

from IPython.display import FileLink, display

source = os.path.join(OUT_DIR, "mastered") if ZIP_WHAT == "mastered" else OUT_DIR
if not os.path.isdir(source):
    raise RuntimeError(f"Nothing in {source} yet. Run step 6 first, or set ZIP_WHAT = \"everything\".")
zip_path = shutil.make_archive("/kaggle/working/my_songs", "zip", source)
print(f"{os.path.getsize(zip_path) / 1e6:.0f} MB. Click to download, or use the Output panel on the right:")
os.chdir("/kaggle/working")
display(FileLink(os.path.basename(zip_path)))

## 🛠️ Troubleshooting

- **Song stuck at "Generating" with no progress**: if you switched to `LM_BACKEND = "vllm"`, set it back to `"pt"` in step 1, then run steps 1 and 3 again.
- **"Generation produced NaN or Inf latents"**: step 2 should have printed "Patched ACE-Step: the song model runs in float32 on this GPU." If it didn't, run step 2 again, then step 3.
- **"Out of memory" in step 4 or 5**: use fewer `TAKES` or a shorter `DURATION`. If it keeps happening, set `LM_MODEL = "acestep-5Hz-lm-0.6B"` in step 1, run steps 1–3 again (step 2 then only downloads the smaller planner), and continue.
- **The engine crashed mid-song**: run step 3 again. Your earlier takes stay in `/kaggle/working/songs`.
- **Not enough disk**: step 1 switches to the 2B model automatically when XL won't fit.
- **New session**: Kaggle wipes the machine between sessions, so run steps 1–3 again. Download your songs (step 7) before you stop.

**License:** ACE-Step 1.5 code and weights are MIT. The [model card](https://huggingface.co/ACE-Step/Ace-Step1.5) says the music you generate can be used commercially. Use only lyrics you wrote or have rights to.
Kaggle's free GPUs are for personal use. Don't run this notebook as a service for other people.